In [1]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate

In [2]:
#constants for 11B nuclei
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)
wkhz = w0*10**3 # LArmor frequency in Hz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*(wkhz))

# Coefficient for LHQ (cluster 1) from ASICS (in Hz)
# A_coeff = [-1.730215,-2.744504,-3.396561]
# B_coeff = [1.251296,2.561384,-1.133846]
# C_coeff = [3.573296,0.986618,-0.473004]
# D_coeff = [-0.060956,-0.376258,-0.980912]
# E_coeff = [-0.037878,-0.579924,-1.166911]

#Coefficient for LHQ (cluster 2) from ASICS (in kHz)
# A_coeff = [-2.237,-2.631,-3.326]
# B_coeff = [1.436,2.462,-1.178]
# C_coeff = [-3.527,0.272,-0.705]
# D_coeff = [0.223,-0.402,-0.926]
# E_coeff = [0.005, -0.526,-1.129]

# Coefficient for RHQ (cluster 1) from ASICS (in kHz)
A_coeff = [-0.902953, -3.845474, -1.865134]
B_coeff = [-1.814913, -0.065823, -1.927795]
C_coeff = [0.661836, 0.820338, 2.050566]
D_coeff = [-0.373541, 0.510270, 0.604851]
E_coeff = [-0.493618, 1.537412, -0.045552]

# Coefficient for RHQ (cluster 2) from ASICS (in kHz)
# A_coeff = [-2.767572, -2.045625, -2.035060]
# B_coeff = [-0.021837, 0.688740, -0.697897]
# C_coeff = [-2.611411, -1.907098, 0.778273]
# D_coeff = [0.910820, -0.481464, -0.288617]
# E_coeff = [0.405120, 0.489621, 0.805603]


In [3]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    
    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, sorted_eigenvectors,  avg_tensor, eigenvalues, eigenvectors

In [5]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-167.363682076023, 167.363682076023], Ayz: [-168.267216985673, 168.267216985673]
Azz - Axx: [-493.009129220316, 493.009129220316], Axz: [177.911921829874, -177.911921829874]
Ayy - Axx: [-371.788814125102, 371.788814125102], Axy: [-6.99005956290305, 6.99005956290305]


In [6]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print('Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): \n ', combinations)

Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): 
  [(-493.009129220316, -371.788814125102, -167.363682076023), (-493.009129220316, -371.788814125102, 167.363682076023), (-493.009129220316, 371.788814125102, -167.363682076023), (-493.009129220316, 371.788814125102, 167.363682076023), (493.009129220316, -371.788814125102, -167.363682076023), (493.009129220316, -371.788814125102, 167.363682076023), (493.009129220316, 371.788814125102, -167.363682076023), (493.009129220316, 371.788814125102, 167.363682076023)]


In [7]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = -np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = -np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = -np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = -Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = -Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = -Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])


print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)




Axx1: [288.265981115139, 288.265981115139, 40.4067716984049, 40.4067716984049, -40.4067716984049, -40.4067716984049, -288.265981115139, -288.265981115139]
Axx2: [303.647103442075, 192.071315391393, -192.071315391393, -303.647103442075, 303.647103442075, 192.071315391393, -192.071315391393, -303.647103442075]
Axx3: [272.884858788203, 384.460646838885, 272.884858788203, 384.460646838885, -384.460646838885, -272.884858788203, -384.460646838885, -272.884858788203]
Ayy1: [-83.5228330099624, -83.5228330099624, 412.195585823506, 412.195585823506, -412.195585823506, -412.195585823506, 83.5228330099624, 83.5228330099624]
Ayy2: [-68.1417106830262, -179.717498733708, 179.717498733708, 68.1417106830262, -68.1417106830262, -179.717498733708, 179.717498733708, 68.1417106830262]
Ayy3: [-52.7605883560901, -275.912164457454, -52.7605883560901, -275.912164457454, 275.912164457454, 52.7605883560901, 275.912164457454, 52.7605883560901]
Azz1: [-204.743148105177, -204.743148105177, -452.602357521911, -452.6

In [17]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*wkhz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*wkhz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*wkhz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)



[{Ayy_s: 5.15137412934570e-7, Azz_s: 9.03594501385399e-6, Ayz_s: -7.95436253052049e-7}, {Axx_s: 1.42568490956544e-5, Azz_s: 1.01816322355205e-5, Axz_s: -3.23914827566242e-6}, {Axx_s: 1.44400292113122e-5, Ayy_s: -1.21884463134403e-8, Axy_s: 3.28906543458224e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 1.43484391534833e-5 2.51474483310565e-7 9.60878862468725e-6 -7.95436253052049e-7 -3.23914827566242e-6 3.28906543458224e-6


In [9]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;

print('Quadrupolar tensor (tenon frame): \n', Q_T, '\n')

#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T)

Quadrupolar tensor (tenon frame): 
 [[-288.26598112    6.99005956 -177.91192183]
 [   6.99005956   68.14171068  168.26721699]
 [-177.91192183  168.26721699  220.12427043]] 

Chemical Shift tensor (tenon frame): 
 [[ 1.43484392e-05  3.28906543e-06 -3.23914828e-06]
 [ 3.28906543e-06  2.51474483e-07 -7.95436253e-07]
 [-3.23914828e-06 -7.95436253e-07  9.60878862e-06]]


In [10]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS

sorted_eigenvalues_Q, sorted_eigenvectors_Q, quad_avg, eigenvalues_Q, eigenvectors_Q = sort_eigenvalues(Q_T)
# print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_Q, '\n') 

Vyy = (sorted_eigenvalues_Q[0])*(2*Ispin*(2*Ispin - 1)) 
Vxx = (sorted_eigenvalues_Q[1])*(2*Ispin*(2*Ispin - 1))
Vzz = (sorted_eigenvalues_Q[2])*(2*Ispin*(2*Ispin - 1)) 

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================ \n')

#Calculate CSA Tensor in PAS

sorted_eigenvalues_csa, sorted_eigenvectors_csa, csa_avg, eigenvalues_csa, eigenvectors_csa = sort_eigenvalues(CS_T)
# print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_csa, '\n') 

csyy = -(sorted_eigenvalues_csa[0]) 
csxx = -(sorted_eigenvalues_csa[1])
cszz = -(sorted_eigenvalues_csa[2]) 

print('CSA Tensor Components δyy, δxx, δzz: \n', csyy, csxx, cszz)



 Unsorted Eigenvalues:
 [-352.93901245  362.71810571   -9.77909326] 

 Unsorted Eigenvectors:
 [[ 0.93128137 -0.22662214  0.28523221]
 [-0.14840934  0.47903611  0.86515841]
 [ 0.33270058  0.84803704 -0.41248456]] 

Sorted Eigenvalues: 
 [  -9.77909326 -352.93901245  362.71810571] 

Sorted Eigenvectors: 
 [[ 0.28523221  0.93128137 -0.22662214]
 [ 0.86515841 -0.14840934  0.47903611]
 [-0.41248456  0.33270058  0.84803704]] 

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -58.67455954156101 -2117.6340747109593 2176.3086342525194

 Unsorted Eigenvalues:
 [ 1.66559634e-05  8.03149686e-06 -4.78758005e-07] 

 Unsorted Eigenvectors:
 [[-0.88204856 -0.41930128 -0.2148878 ]
 [-0.19758883 -0.08485011  0.97660592]
 [ 0.42772536 -0.90387327  0.00800733]] 

Sorted Eigenvalues: 
 [ 8.03149686e-06 -4.78758005e-07  1.66559634e-05] 

Sorted Eigenvectors: 
 [[-0.41930128 -0.2148878  -0.88204856]
 [-0.08485011  0.97660592 -0.19758883]
 [-0.90387327  0.00800733  0.42772536]] 

CSA Tensor Components δyy, δxx,

In [11]:
#Quadrupolar tensor parameters
cq = Vzz/10**3
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
iso_cs = np.mean([cszz, csyy, csxx]) 
csa = cszz - iso_cs

etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs*10**6], ['csa (ppm)', csa*10**6], ['etas', etas] ] #converting Hz to ppm (Should be multiplied by 10**6)
print(tabulate(table, headers=['Quantity', 'Fit Value']))

Quantity        Fit Value
------------  -----------
cq (MHz)         2.17631
etaq             0.946079
iso_cs (ppm)    -8.06957
csa (ppm)       -8.5864
etas             0.991132


In [12]:
print(sorted_eigenvectors_Q)

a_Q, b_Q, g_Q = get_euler_angles(sorted_eigenvectors_Q)
print("Calculated Euler angles (degrees):")
print('alpha:', a_Q, 'beta:', b_Q, 'gamma:', g_Q,'\n')

[[ 0.28523221  0.93128137 -0.22662214]
 [ 0.86515841 -0.14840934  0.47903611]
 [-0.41248456  0.33270058  0.84803704]]
Calculated Euler angles (degrees):
alpha: -38.888850805359226 beta: 32.001195912423206 gamma: 64.68213065321132 



In [13]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(sorted_eigenvectors_Q), (sorted_eigenvectors_csa))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -36.31037336199805 chi: 62.09762144874483 xi: -47.331453473470894 



In [14]:
A = [1, 2, 3]
B = [5 * i for i in A]

print(A, B)

[1, 2, 3] [5, 10, 15]


In [15]:
# Find Rotation Matrix and Rotation angles

# *************** Calculation for CSA ************************
# Quadrupolar Tensor in PAS
Q_PAS = np.zeros((3,3))
Q_PAS[0,0] = Vxx/(2*Ispin*(2*Ispin - 1));
Q_PAS[1,1] = Vyy/(2*Ispin*(2*Ispin - 1));
Q_PAS[2,2] = Vzz/(2*Ispin*(2*Ispin - 1));

#CSA tensor in PAS
CS_PAS = np.zeros((3,3))
CS_PAS[0,0] = -csxx; 
CS_PAS[1,1] = -csyy;
CS_PAS[2,2] = -cszz;
print('Calculation for CSA Tensor: \n')

# Find eigenvalues and eigenvectors of original matrix

eigenvalues, eigenvectors = np.linalg.eig(CS_PAS) 
print('Eigenvalues of CSA (PAS) tensor \n', eigenvalues, '\n')
print('Eigenvectors of CSA (PAS) tensor \n', eigenvectors, '\n')
# Calculate the eigenvalues of the rotated matrix A_rot

eigenvalues_rot, eigenvectors_rot = np.linalg.eig(CS_T)
print('Eigenvalues of CSA (Tenon) tensor \n', eigenvalues_rot, '\n')
print('Eigenvectors of CSA (Tenon) tensor \n', eigenvectors_rot, '\n')

b = np.degrees(np.arccos(CS_T[2,2]))
a = np.degrees(np.arctan(CS_T[2,1]/CS_T[2,0]))
g = np.degrees(np.arctan(-CS_T[1,2]/CS_T[0,2]))
print("Calculated Euler angles (degrees):")
print(a, b, g,'\n')






Calculation for CSA Tensor: 

Eigenvalues of CSA (PAS) tensor 
 [-4.78758005e-07  8.03149686e-06  1.66559634e-05] 

Eigenvectors of CSA (PAS) tensor 
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]] 

Eigenvalues of CSA (Tenon) tensor 
 [ 1.66559634e-05  8.03149686e-06 -4.78758005e-07] 

Eigenvectors of CSA (Tenon) tensor 
 [[-0.88204856 -0.41930128 -0.2148878 ]
 [-0.19758883 -0.08485011  0.97660592]
 [ 0.42772536 -0.90387327  0.00800733]] 

Calculated Euler angles (degrees):
13.797082720127621 89.99944945696556 -13.797082720127621 

